# Notebook for benchmark

In [1]:
#!/usr/bin/env python
# coding: utf-8

# # PathSeeker Benchmark
# 
# Performance for PathSeeker for model and non-model organisms.
# Metrics:
# 1. Reaction coverage
# 2. Compound coverage
# 3. Equation correctness
# 4. Integration metric

import pandas as pd

# ---
# ## 1. Load Data
predicted = pd.read_csv("../output/matched_metabolites_reactions_all.csv")
predicted.head()


,Reaction,Compound,Role,Origin,equation
0,R01404,C00173,substrates,proteomics,C00173 + C00006 <=> C00693 + C00005 + C00080
1,R01404,C00693,products,proteomics,C00173 + C00006 <=> C00693 + C00005 + C00080
2,R01624,C00024,substrates,proteomics,C00024 + C00229 <=> C00010 + C03939
3,R01624,C00010,products,proteomics,C00024 + C00229 <=> C00010 + C03939
4,R01626,C00083,substrates,proteomics,C00083 + C00229 <=> C00010 + C01209


# Step 0 - KEGG Global Reactions & Reaction Coverage

### Using the Global KEGG Reaction List
- We use the **global KEGG reaction list** when a complete reference of all known reactions is needed.  
- This allows us to ask:  
  *“Out of all possible reactions in KEGG, how many did PathSeeker recover?”*  
  → This shows the **breadth of the method**.

### Limitations / Interpretation
- The global list does **not** imply that all 12k reactions are biologically relevant for *T. versicolor* (or *S. cerevisiae*).  
- Coverage calculated against the global list is **conservative/expansive**:  
  - Low values do **not necessarily** indicate a failure of PathSeeker.  
  - Many KEGG reactions may simply not apply to the organism.


- Reported **two coverage metrics** in the manuscript:
  1. **Coverage vs global KEGG** (full list of known reactions)  
  2. **Coverage vs organism-specific pathway list** 

### Metrics Calculated
Given the predicted reactions from PathSeeker and the KEGG global reference:

- **TP** = # reactions predicted ∩ KEGG_global  
- **FP** = # reactions predicted \ KEGG_global  
- **FN** = # KEGG_global \ predicted (can be very large with the global list)  
- **Precision** = TP / (TP + FP)  
- **Recall / Coverage** = TP / (TP + FN)  
- **F1-score** = 2 × (Precision × Recall) / (Precision + Recall)  
- **Integration metric** = % of predicted reactions with Origin == "both"  
- **Equation correctness** = whether reaction ID exists in KEGG and whether predicted equation matches exactly (or normalized comparison)


# 1. Reaction coverage (coverage vs global KEGG)

In [1]:
# Benchmark using global KEGG reactions as reference
import pandas as pd
from collections import Counter

# ===============================
# 1. Paths (ajuste se necessário)
# ===============================
matched_csv = "../output/matched_metabolites_reactions_all.csv"
kegg_global_csv = "../reference/kegg_reactions_sce.csv"  # your global KEGG file

# ===============================
# 2. Load
# ===============================
df_matched = pd.read_csv(matched_csv, dtype=str).fillna("")
df_kegg = pd.read_csv(kegg_global_csv, dtype=str).fillna("")

# Remover prefixo "EQUATION " (se existir) e espaços extras
if "equation" in df_kegg.columns:
    df_kegg["equation"] = (
        df_kegg["equation"]
        .astype(str)
        .str.replace(r"^EQUATION\s*", "", regex=True)
        .str.strip()
    )

# ===============================
# 3. Unique reaction sets
# ===============================
predicted_reactions = set(df_matched["Reaction"].unique())
kegg_reactions = set(df_kegg["Reaction"].unique())

TP = predicted_reactions & kegg_reactions
FP = predicted_reactions - kegg_reactions
FN = kegg_reactions - predicted_reactions

tp, fp, fn = len(TP), len(FP), len(FN)
total_pred, total_kegg = len(predicted_reactions), len(kegg_reactions)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("=== Reaction-level summary (versus KEGG global) ===")
print(f"Predicted reactions (unique): {total_pred}")
print(f"KEGG reactions (global): {total_kegg}")
print(f"TP (pred ∩ KEGG): {tp}")
print(f"FP (pred \\ KEGG): {fp}")
print(f"FN (KEGG \\ pred): {fn}")
print(f"Precision: {precision:.4f}")
print(f"Recall (coverage): {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# ===============================
# 4. Integration metric
# ===============================
reaction_origins = {}
for rxn, group in df_matched.groupby("Reaction"):
    origins = set()
    for v in group["Origin"].astype(str).unique():
        if pd.isna(v) or v == "":
            continue
        parts = [x.strip().lower() for x in v.split(",") if x.strip()]
        origins.update(parts)
    if "both" in origins or ("proteomics" in origins and "metabolomics" in origins):
        reaction_origins[rxn] = "both"
    elif "proteomics" in origins:
        reaction_origins[rxn] = "proteomics"
    elif "metabolomics" in origins:
        reaction_origins[rxn] = "metabolomics"
    else:
        reaction_origins[rxn] = "unknown"

predicted_origin_counts = Counter(
    reaction_origins[rxn] for rxn in predicted_reactions if rxn in reaction_origins
)
both_count = predicted_origin_counts.get("both", 0)
integration_pct = both_count / total_pred * 100 if total_pred > 0 else 0.0
print(f"\nIntegration (predicted reactions with 'both' support): {both_count} / {total_pred} = {integration_pct:.2f}%")

# ===============================
# 5. Equation correctness
# ===============================
kegg_eq_dict = dict(zip(df_kegg["Reaction"], df_kegg["equation"]))
correct_eq, checked = 0, 0

for rxn in predicted_reactions:
    pred_eqs = df_matched.loc[df_matched["Reaction"] == rxn, "equation"].unique()
    kegg_eq = kegg_eq_dict.get(rxn, "").strip()
    if not kegg_eq:
        continue
    for pe in pred_eqs:
        if str(pe).strip() == kegg_eq:
            correct_eq += 1
            break
    checked += 1

eq_correct_pct = (correct_eq / checked * 100) if checked > 0 else None
print(f"\nEquation correctness (exact match) over {checked} reactions with KEGG equation available: {correct_eq} = {eq_correct_pct:.2f}%")


=== Reaction-level summary (versus KEGG global) ===
Predicted reactions (unique): 3402
KEGG reactions (global): 12312
TP (pred ∩ KEGG): 3402
FP (pred \ KEGG): 0
FN (KEGG \ pred): 8910
Precision: 1.0000
Recall (coverage): 0.2763
F1-score: 0.4330

Integration (predicted reactions with 'both' support): 212 / 3402 = 6.23%

Equation correctness (exact match) over 3402 reactions with KEGG equation available: 3402 = 100.00%


## Results

### Predicted reactions (unique): **3,402**
- PathSeeker predicted 3,402 different reactions for the organism.

### KEGG reactions (global): **12,312**
- Reference universe (background), all reactions available in KEGG.

### TP = **3,402**
- All predictions match existing KEGG reactions (no extras).  
- This explains **Precision = 1.0** (no false positives).

### FN = **8,910**
- Reactions from KEGG that did not appear in your prediction.

### Recall (coverage) = **0.2763 (~27.6%)**
- The organism covers a little more than 1/4 of the global universe.  
- This is expected, since no organism covers everything — each only has a subset.

### F1-score = **0.433**
- Combines precision and coverage.  
- Intermediate value, but reasonable for non-model organisms.

### Integration = **6.23%**
- Only 212 of the 3,402 reactions have “dual” support (e.g., proteomics + transcriptomics, or multiple lines of evidence).  
- This measures confidence in the predictions.

### Equation correctness = **100%**
- All predicted reaction equations perfectly match KEGG after cleaning.

---

### Summary
- **High precision (100%)** → PathSeeker is not inventing reactions outside KEGG.  
- **Moderate coverage (27%)** → Normal, since the global universe is huge.  
- Comparing this number between model vs. non-model organisms will be the core of the benchmark.


# Reaction coverage (coverage vs Reactions specific to *S. cerevisiae*)

In [9]:
# Benchmark using global KEGG reactions as reference
import pandas as pd
from collections import Counter

# ===============================
# 1. Paths (ajuste se necessário)
# ===============================
matched_csv = "../output/matched_metabolites_reactions_all.csv"
kegg_global_csv = "../reference/kegg_reactions_sce_specific.csv"  # your global KEGG file

# ===============================
# 2. Load
# ===============================
df_matched = pd.read_csv(matched_csv, dtype=str).fillna("")
df_kegg = pd.read_csv(kegg_global_csv, dtype=str).fillna("")

# Remover prefixo "EQUATION " (se existir) e espaços extras
if "equation" in df_kegg.columns:
    df_kegg["equation"] = (
        df_kegg["equation"]
        .astype(str)
        .str.replace(r"^EQUATION\s*", "", regex=True)
        .str.strip()
    )

# ===============================
# 3. Unique reaction sets
# ===============================
predicted_reactions = set(df_matched["Reaction"].unique())
kegg_reactions = set(df_kegg["Reaction"].unique())

TP = predicted_reactions & kegg_reactions
FP = predicted_reactions - kegg_reactions
FN = kegg_reactions - predicted_reactions

tp, fp, fn = len(TP), len(FP), len(FN)
total_pred, total_kegg = len(predicted_reactions), len(kegg_reactions)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("=== Reaction-level summary (versus KEGG S. cerevisiae) ===")
print(f"Predicted reactions (unique): {total_pred}")
print(f"KEGG reactions (S. cerevisiae): {total_kegg}")
print(f"TP (pred ∩ KEGG): {tp}")
print(f"FP (pred \\ KEGG): {fp}")
print(f"FN (KEGG \\ pred): {fn}")
print(f"Precision: {precision:.4f}")
print(f"Recall (coverage): {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# ===============================
# 4. Integration metric
# ===============================
reaction_origins = {}
for rxn, group in df_matched.groupby("Reaction"):
    origins = set()
    for v in group["Origin"].astype(str).unique():
        if pd.isna(v) or v == "":
            continue
        parts = [x.strip().lower() for x in v.split(",") if x.strip()]
        origins.update(parts)
    if "both" in origins or ("proteomics" in origins and "metabolomics" in origins):
        reaction_origins[rxn] = "both"
    elif "proteomics" in origins:
        reaction_origins[rxn] = "proteomics"
    elif "metabolomics" in origins:
        reaction_origins[rxn] = "metabolomics"
    else:
        reaction_origins[rxn] = "unknown"

predicted_origin_counts = Counter(
    reaction_origins[rxn] for rxn in predicted_reactions if rxn in reaction_origins
)
both_count = predicted_origin_counts.get("both", 0)
integration_pct = both_count / total_pred * 100 if total_pred > 0 else 0.0
print(f"\nIntegration (predicted reactions with 'both' support): {both_count} / {total_pred} = {integration_pct:.2f}%")

# ===============================
# 5. Equation correctness
# ===============================
kegg_eq_dict = dict(zip(df_kegg["Reaction"], df_kegg["equation"]))
correct_eq, checked = 0, 0

for rxn in predicted_reactions:
    pred_eqs = df_matched.loc[df_matched["Reaction"] == rxn, "equation"].unique()
    kegg_eq = kegg_eq_dict.get(rxn, "").strip()
    if not kegg_eq:
        continue
    for pe in pred_eqs:
        if str(pe).strip() == kegg_eq:
            correct_eq += 1
            break
    checked += 1

eq_correct_pct = (correct_eq / checked * 100) if checked > 0 else None
print(f"\nEquation correctness (exact match) over {checked} reactions with KEGG equation available: {correct_eq} = {eq_correct_pct:.2f}%")


=== Reaction-level summary (versus KEGG S. cerevisiae) ===
Predicted reactions (unique): 3402
KEGG reactions (S. cerevisiae): 1512
TP (pred ∩ KEGG): 1097
FP (pred \ KEGG): 2305
FN (KEGG \ pred): 415
Precision: 0.3225
Recall (coverage): 0.7255
F1-score: 0.4465

Integration (predicted reactions with 'both' support): 212 / 3402 = 6.23%

Equation correctness (exact match) over 1097 reactions with KEGG equation available: 1097 = 100.00%


# 2. Compound coverage

In [18]:
# benchmark_compound_coverage.py
import pandas as pd

def compound_coverage(metabolomics_file="../output/metabolomics_with_C_numbers_curated.xlsx",
                      predicted_file="../output/matched_metabolites_reactions_all.csv"):

    # 1. Carregar metabolômica experimental (Excel)
    metabolomics = pd.read_excel(metabolomics_file)
    detected = set(metabolomics["KEGG_C_number"].dropna().unique())

    # 2. Carregar predicted compounds (CSV normal)
    predicted = pd.read_csv(predicted_file)
    mapped = set(predicted["Compound"].dropna().unique())

    # 3. Interseção
    overlap = detected & mapped
    coverage = len(overlap) / len(detected) * 100 if len(detected) > 0 else 0

    result = {
        "Total detected (metabolomics)": len(detected),
        "Total mapped (PathSeeker)": len(mapped),
        "Overlap (detected ∩ mapped)": len(overlap),
        "Compound coverage (%)": coverage
    }

    print("=== Compound coverage summary ===")
    for k, v in result.items():
        if isinstance(v, float):
            print(f"{k}: {v:.2f}")
        else:
            print(f"{k}: {v}")

    return result


if __name__ == "__main__":
    compound_coverage()


=== Compound coverage summary ===
Total detected (metabolomics): 181
Total mapped (PathSeeker): 1192
Overlap (detected ∩ mapped): 163
Compound coverage (%): 90.06


# Compound Coverage Explanation

**Definitions:**

- **Detected metabolites (experimental)**: metabolites actually observed in the metabolomics experiment.  
  Example: 181 compounds detected.

- **Mapped metabolites (PathSeeker predictions)**: metabolites associated with reactions predicted by PathSeeker in the metabolic model.  
  Example: 1192 compounds mapped.

- **Overlap (detected ∩ mapped)**: metabolites both detected experimentally and present in the PathSeeker predictions.  
  Example: 163 compounds.

**Interpretation:**

- The compound coverage measures how many of the experimentally detected metabolites are also captured by PathSeeker predictions:

$$
\text{Compound coverage (\%)} = \frac{|\text{Detected} \cap \text{Mapped}|}{|\text{Detected}|} \times 100
= \frac{163}{181} \times 100 \approx 90\%
$$

- The mapped set can be larger than the detected set because PathSeeker predicts **all metabolites associated with the reactions in the model**, not only those measured experimentally.

**Diagram (Venn-style):**



           ┌───────────────┐
           │               │
           │   Mapped      │
           │  metabolites  │
           │   (1192)      │
           │        ┌──────┴─────┐
           │        │            │
           │        │  Overlap   │
           │        │  (163)     │
           │        │            │
           │        └────────────┘
           │
           │
           └───────────────┐
                           │
                           │ Detected metabolites
                           │  (181 total)



- **Key point:** The overlap shows that PathSeeker captures ~90% of experimentally detected metabolites, indicating good coverage of your metabolomics dataset.


# 3. Equation correctness

In [3]:
import pandas as pd

def clean_equation(eq: str) -> str:
    """
    Normaliza equações para comparação:
    - Remove prefixo 'EQUATION' se existir
    - Tira espaços extras
    """
    if pd.isna(eq):
        return ""
    return eq.replace("EQUATION", "").strip()

def equation_correctness(predicted_file="../output/matched_metabolites_reactions_all.csv",
                         kegg_file="../reference/kegg_reactions_sce.csv"):
    """
    Calcula a porcentagem de equações corretas comparando PathSeeker vs KEGG.
    Compara strings completas, mantendo +, <=> e ordem.
    """
    # Carregar arquivos
    predicted = pd.read_csv(predicted_file, dtype=str).fillna("")
    kegg_ref  = pd.read_csv(kegg_file, dtype=str).fillna("")
    
    # Normalizar equações
    predicted["equation"] = predicted["equation"].apply(clean_equation)
    kegg_ref["equation"]  = kegg_ref["equation"].apply(clean_equation)
    
    correct = 0
    total   = 0
    
    for rxn in predicted["Reaction"].unique():
        pred_eqs = predicted.loc[predicted["Reaction"] == rxn, "equation"].unique()
        kegg_eqs = kegg_ref.loc[kegg_ref["Reaction"] == rxn, "equation"].unique()
        
        if len(kegg_eqs) == 0:
            continue
        
        # Checar se alguma equação prevista bate exatamente com KEGG
        if any(pe.strip() in [ke.strip() for ke in kegg_eqs] for pe in pred_eqs):
            correct += 1
        
        total += 1
    
    correctness = (correct / total * 100) if total > 0 else 0
    print(f"✅ Equation correctness: {correctness:.2f}% ({correct}/{total} reactions)")
    return correctness

# Rodar
equation_correctness()



✅ Equation correctness: 100.00% (3402/3402 reactions)


100.0

# 4. Integration metric

In [23]:
# Integration metric: % de reações suportadas por BOTH (proteomics + metabolomics)
integration = (
    len(predicted[predicted["Origin"] == "both"]["Reaction"].unique()) / 
    len(predicted["Reaction"].unique()) * 100
)
print(f"Integration metric: {integration:.2f}%")


Integration metric: 6.23%


### Integration Metric

The **integration metric** measures the proportion of predicted reactions that are supported simultaneously by both proteomics and metabolomics evidence (`Origin = "both"`).

**Formula:**


$$
\text{Integration (\%)} \;=\; \frac{\text{\# reactions with Origin = "both"}}{\text{\# total predicted reactions}} \times 100
$$

- **Reactions with `Origin = "both"`**: number of reactions with experimental support from both proteomics and metabolomics.  
- **Total predicted reactions**: total number of unique reactions predicted by PathSeeker.  

**Example:**  
If 3,402 reactions are predicted and 212 are supported by both data types, then:

$$
\frac{212}{3402} \times 100 \approx 6.23\%
$$

This metric provides an additional layer of confidence in the reconstructed network, as reactions supported by multiple omics sources are more likely to represent true biological activity.


# 5. Benchmark Overview – PathwaySeeker Predictions

In [25]:
from pathlib import Path
import pandas as pd

# 1️⃣ Carregar CSV
matched_csv = "../output/matched_metabolites_reactions_all.csv"
df = pd.read_csv(matched_csv, dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]
df['Role'] = df['Role'].str.lower().str.rstrip('s')
df['Origin'] = df['Origin'].fillna("")
if 'equation' not in df.columns and 'Equation' in df.columns:
    df['equation'] = df['Equation']

# 2️⃣ Criar edges
edges = []
from itertools import product

grouped = df.groupby('Reaction')
for reaction, group in grouped:
    subs = group[group['Role']=='substrate'][['Compound','Origin']].drop_duplicates()
    prods = group[group['Role']=='product'][['Compound','Origin']].drop_duplicates()
    for s_rec, p_rec in product(subs.to_dict('records'), prods.to_dict('records')):
        if s_rec['Compound'] == p_rec['Compound']:
            continue
        origins = set()
        for x in [s_rec['Origin'], p_rec['Origin']]:
            if x:
                origins |= set([z.strip() for z in x.split(',')])
        if 'both' in origins or ('proteomics' in origins and 'metabolomics' in origins):
            origin_edge = 'both'
        elif 'proteomics' in origins:
            origin_edge = 'proteomics'
        elif 'metabolomics' in origins:
            origin_edge = 'metabolomics'
        else:
            origin_edge = 'unknown'
        edges.append({'Reaction': reaction, 'source': s_rec['Compound'], 'target': p_rec['Compound'], 'origin_edge': origin_edge})

edges_df = pd.DataFrame(edges).drop_duplicates(subset=['Reaction','source','target'])

# 3️⃣ Calcular métricas
metrics = {
    'total_reactions': df['Reaction'].nunique(),
    'total_compounds': df['Compound'].nunique(),
    'total_edges': len(edges_df),
    'edges_proteomics_only': int((edges_df['origin_edge']=='proteomics').sum()),
    'edges_metabolomics_only': int((edges_df['origin_edge']=='metabolomics').sum()),
    'edges_both': int((edges_df['origin_edge']=='both').sum()),
    'edges_unknown': int((edges_df['origin_edge']=='unknown').sum())
}

metrics


{'total_reactions': 3402,
 'total_compounds': 1192,
 'total_edges': 1897,
 'edges_proteomics_only': 1032,
 'edges_metabolomics_only': 372,
 'edges_both': 493,
 'edges_unknown': 0}

## Benchmark Overview – PathwaySeeker Predictions
### Overview
- **Total reactions (predicted):** 3,402  
- **Total compounds (predicted):** 1,192  
- **Total edges (substrate → product):** 1,897  

### Edge Origins

| Origin | Count | % of total edges |
|--------|-------|----------------|
| Proteomics only | 1,032 | 54.4% |
| Metabolomics only | 372 | 19.6% |
| Both | 493 | 26.0% |
| Unknown | 0 | 0.0% |

**Integration metric:**  
$$
\text{Integration (\%)} = \frac{\text{edges with both support}}{\text{total edges}} \times 100 \approx 26\%
$$

### Diagram: Relationships
Detected Compounds (metabolomics) 

        │
        ▼
Predicted Compounds & Reactions (PathwaySeeker) 

        │
        ▼
Edges (substrate → product)

        │
        ├─ Proteomics only: 1032 edges
        ├─ Metabolomics only: 372 edges
        └─ Both (Integration): 493 edges

### Notes
- Each reaction may have multiple substrates and products, generating multiple edges.  
- The `origin_edge` column indicates the evidence source for each edge.  
- Compound coverage, reaction coverage, and integration metrics are derived from these edges.  


In [11]:
from graphviz import Digraph

# Criar grafo
dot = Digraph('PathSeeker Workflow', format='png')
dot.attr(rankdir='LR', fontsize='12', fontname='Helvetica')

# Inputs
dot.node('P', 'Proteomics Data\n(proteins with KO IDs)', shape='box', style='filled', color='#A6CEE3')
dot.node('M', 'Metabolomics Data\n(metabolites with C-numbers)', shape='box', style='filled', color='#B2DF8A')

# Pipeline: PathSeeker Steps
with dot.subgraph(name='cluster_PS') as c:
    c.attr(style='rounded', color='gray90', label='PathSeeker Pipeline', fontsize='12', fontname='Helvetica')
    c.node('KO', 'KO Mapping & Reaction Recovery', shape='box', style='filled', color='#FDBF6F')
    c.node('RXN_EXP', 'Reaction-to-Compound Expansion', shape='box', style='filled', color='#FFEDA0')
    c.node('INTEG', 'Integration & Matching', shape='box', style='filled', color='#FDBF6F')
    c.node('NET', 'Network Reconstruction', shape='box', style='filled', color='#FFEDA0')
    
    # Conectar subetapas
    c.edge('KO', 'RXN_EXP')
    c.edge('RXN_EXP', 'INTEG')
    c.edge('INTEG', 'NET')

# Output
dot.node('JSON', 'Output JSON\n(graph_all.json)', shape='box', style='filled', color='#FFFFB2')
dot.node('PV', 'PathwayViz\nInteractive Visualization', shape='box', style='filled', color='#FFCCCB')

# Conectar inputs ao pipeline
dot.edge('P', 'KO')
dot.edge('M', 'RXN_EXP')

# Conectar pipeline ao output
dot.edge('NET', 'JSON')
dot.edge('JSON', 'PV')

# Renderizar
dot.render('figure1_pathseeker_workflow', view=True)


'figure1_pathseeker_workflow.png'